# Section B, Part B, Part C, and Part D

Simple guide:
- This notebook is the one we run and submit.
- All code is inside this notebook, single file.
- Part B search formulation, Part C Astar with g and h1 h2, Part D runs BFS vs Astar h1 vs Astar h2 and the csv stuff.

In [1]:
import heapq
import csv
import itertools
import time
import tracemalloc
from collections import deque
from dataclasses import dataclass, field
from pathlib import Path

import cv2
import numpy as np

# Rough layout: Part B is the state and successor stuff, Part C is g h1 h2 and Astar,
# Part D is BFS vs Astar with h1 vs h2 plus the CSV, same likelihood function for all of it.

try:
    from skimage.transform import hough_line, hough_line_peaks
    HAS_SKIMAGE = True
except Exception:
    HAS_SKIMAGE = False

# Use a smaller working size for faster experiments.
target_height = 1400
target_width = 1050


# Likelihood when you have 4 corners, blend edge match on the mask with the A4 ratio prior.
def likelihoodA4(rect, bw, weightA4Prior=0.35, neighborhood=1):
    if rect.shape != (4, 2):
        raise ValueError("rect should be 4x2 array.")
    image_height, image_width = bw.shape
    p_edge = 0.0
    for i in range(4):
        pt1, pt2 = rect[i], rect[(i + 1) % 4]
        if np.allclose(pt1, pt2):
            return 0.0
        x1, y1 = pt1
        x2, y2 = pt2
        n = max(int(2 * max(image_height, image_width)), 1)
        xs = np.rint(np.linspace(x1, x2, n)).astype(int)
        ys = np.rint(np.linspace(y1, y2, n)).astype(int)
        pts = np.unique(np.column_stack((ys, xs)), axis=0)
        if neighborhood > 0:
            expanded = []
            for (r, c) in pts:
                if neighborhood == 1:
                    neigh = [(r, c), (r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)]
                else:
                    neigh = [
                        (r, c), (r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1),
                        (r + 1, c + 1), (r - 1, c + 1), (r + 1, c - 1), (r - 1, c - 1),
                    ]
                expanded.extend(neigh)
            pts = np.unique(np.array(expanded), axis=0)
        inb = (
            (pts[:, 0] >= 0) & (pts[:, 0] < image_height) &
            (pts[:, 1] >= 0) & (pts[:, 1] < image_width)
        )
        pts = pts[inb]
        if len(pts) == 0:
            return 0.0
        p_edge += np.sum(bw[pts[:, 0], pts[:, 1]]) / float(len(pts))
    p_edge /= 4.0

    v1, v2, v3, v4 = rect[0] - rect[1], rect[1] - rect[2], rect[2] - rect[3], rect[3] - rect[0]
    n1, n2, n3, n4 = np.linalg.norm(v1), np.linalg.norm(v2), np.linalg.norm(v3), np.linalg.norm(v4)
    if min(n1, n2, n3, n4) < 1e-6:
        return 0.0
    if n1 > n2:
        ratio1, ratio2 = n1 / n2, n3 / n4
    else:
        ratio1, ratio2 = n2 / n1, n4 / n3
    ratio = (ratio1 + ratio2) / 2.0
    mu, sigma = 297.0 / 210.0, 0.5
    q = np.exp(-((ratio - mu) ** 2) / (2 * sigma ** 2)) / (np.sqrt(2 * np.pi) * sigma)
    return (1 - weightA4Prior) * p_edge + weightA4Prior * q


# N candidate points from Hough lines and intersections, we cap the list so it stays fast.
def preprocess_candidates(filepath, n_hough_peaks=80, angle_l=15, angle_h=75, max_candidates=120):
    img = cv2.imread(str(filepath))
    if img is None:
        raise IOError(f"Cannot read image: {filepath}")

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if gray.shape[0] < gray.shape[1]:
        gray = np.rot90(gray, k=3)

    gray = cv2.resize(gray, (target_width, target_height), interpolation=cv2.INTER_AREA)
    gray_f = gray.astype(np.float32)
    g1 = cv2.GaussianBlur(gray_f, (21, 21), 20)
    g2 = cv2.GaussianBlur(gray_f, (21, 21), 35)
    dog = g1 - g2
    norm = (255.0 * (dog - dog.min()) / (dog.max() - dog.min() + 1e-8)).astype(np.uint8)
    _, binary = cv2.threshold(norm, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    binary = cv2.erode(binary, np.ones((3, 3), np.uint8))
    _edges = cv2.Canny(binary, 50, 150)
    _edges = cv2.dilate(_edges, np.ones((3, 3), np.uint8))
    edge_mask = (_edges > 0).astype(np.float32)

    if HAS_SKIMAGE:
        tested_angles = np.deg2rad(np.arange(-90, 90, 0.5))
        hspace, angles, dists = hough_line(binary > 0, theta=tested_angles)
        _, theta_peaks, dist_peaks = hough_line_peaks(
            hspace, angles, dists, num_peaks=n_hough_peaks, threshold=0.1 * np.max(hspace)
        )
    else:
        edges = cv2.Canny(binary, 50, 150)
        lines = cv2.HoughLines(edges, 1, np.pi / 360.0, 60)
        if lines is None:
            return [], edge_mask, gray.shape[1], gray.shape[0]
        lines = lines[: min(n_hough_peaks, len(lines)), 0, :]
        dist_peaks = lines[:, 0]
        theta_peaks = lines[:, 1]

    deg_thetas = np.rad2deg(theta_peaks)
    keep = [
        not ((th >= -angle_h and th <= -angle_l) or (th >= angle_l and th <= angle_h))
        for th in deg_thetas
    ]
    theta_peaks = theta_peaks[keep]
    dist_peaks = dist_peaks[keep]

    image_h, image_w = gray.shape
    points = []
    for i in range(len(dist_peaks)):
        rho1, theta1 = dist_peaks[i], theta_peaks[i]
        for j in range(i + 1, len(dist_peaks)):
            rho2, theta2 = dist_peaks[j], theta_peaks[j]
            if np.isclose(theta1, theta2, atol=1e-6):
                continue
            A = np.array([[np.cos(theta1), np.sin(theta1)], [np.cos(theta2), np.sin(theta2)]])
            b = np.array([rho1, rho2])
            try:
                x_sol, y_sol = np.linalg.solve(A, b)
            except np.linalg.LinAlgError:
                continue

            # Slightly wider bounds so we do not over-filter real corners.
            if not (image_w * 0.05 < x_sol < image_w * 0.95):
                continue
            if not (image_h * 0.05 < y_sol < image_h * 0.90):
                continue
            points.append((int(round(x_sol)), int(round(y_sol))))

    candidates = list(set(points))

    # Simple fallback: if intersections are empty, use corner features.
    if len(candidates) == 0:
        corners = cv2.goodFeaturesToTrack(binary, maxCorners=max_candidates, qualityLevel=0.01, minDistance=10)
        if corners is not None:
            candidates = [(int(c[0][0]), int(c[0][1])) for c in corners]

    if len(candidates) > max_candidates:
        candidates = sorted(candidates, key=lambda p: (p[1], p[0]))
        step = max(1, len(candidates) // max_candidates)
        candidates = candidates[::step][:max_candidates]
    return candidates, edge_mask, image_w, image_h


# Part B successor rules, about right angles and rough A4 shape when you have 3 corners.
def _angle(v1, v2):
    n1 = np.linalg.norm(v1)
    n2 = np.linalg.norm(v2)
    if n1 < 1e-6 or n2 < 1e-6:
        return 180.0
    c = np.clip(np.dot(v1, v2) / (n1 * n2), -1.0, 1.0)
    return np.degrees(np.arccos(c))


def valid_next(state, pt, image_w, image_h):
    if pt in state:
        return False
    k = len(state)
    if k >= 2:
        p0 = np.array(state[-2], dtype=float)
        p1 = np.array(state[-1], dtype=float)
        p2 = np.array(pt, dtype=float)
        ang = _angle(p0 - p1, p2 - p1)
        if not (75.0 <= ang <= 105.0):
            return False

    if k == 3:
        p0, p1, p2 = [np.array(p, dtype=float) for p in state]
        p3 = np.array(pt, dtype=float)
        e1 = np.linalg.norm(p1 - p0)
        e2 = np.linalg.norm(p2 - p1)
        e3 = np.linalg.norm(p3 - p2)
        e4 = np.linalg.norm(p0 - p3)
        if min(e1, e2, e3, e4) < 1e-6:
            return False
        long_side = (max(e1, e2) + max(e3, e4)) / 2.0
        short_side = (min(e1, e2) + min(e3, e4)) / 2.0
        ratio = long_side / short_side
        if not (1.25 <= ratio <= 1.60):
            return False
        area_ratio = (e1 * e2) / float(image_w * image_h)
        if not (0.03 <= area_ratio <= 0.35):
            return False

    return True


def successors(state, candidates, image_w, image_h):
    # Which points can be the next corner in order.
    return [pt for pt in candidates if valid_next(state, pt, image_w, image_h)]


# Part C cost g, cost_option 1 is uniform steps, 2 adds rectangle deviation, pick one for Astar below.
def g_uniform(state):
    return float(len(state))


def _rectangle_deviation(rect):
    p = np.array(rect, dtype=float)
    v1 = p[1] - p[0]
    v2 = p[2] - p[1]
    v3 = p[3] - p[2]
    v4 = p[0] - p[3]
    n1, n2, n3, n4 = np.linalg.norm(v1), np.linalg.norm(v2), np.linalg.norm(v3), np.linalg.norm(v4)
    if min(n1, n2, n3, n4) < 1e-6:
        return 1.0

    a1 = _angle(-v1, v2)
    a2 = _angle(-v2, v3)
    angle_err = (abs(a1 - 90.0) + abs(a2 - 90.0)) / 180.0

    long_side = (max(n1, n2) + max(n3, n4)) / 2.0
    short_side = (min(n1, n2) + min(n3, n4)) / 2.0
    ratio = long_side / short_side
    ratio_err = abs(ratio - (297.0 / 210.0)) / (297.0 / 210.0)
    return 0.5 * angle_err + 0.5 * ratio_err


def g_penalty(state):
    if len(state) < 4:
        return float(len(state))
    return float(len(state)) + _rectangle_deviation(state)


def h1(state):
    # h1, corners left to place, write the proof in the report.
    return float(4 - len(state))


def _edge_confidence(state, bw_image, image_w, image_h):
    if len(state) < 2:
        return 0.5
    total_on_edge = 0
    total_sampled = 0
    for i in range(len(state) - 1):
        x1, y1 = state[i]
        x2, y2 = state[i + 1]
        n = max(int(np.hypot(x2 - x1, y2 - y1)), 1)
        xs = np.rint(np.linspace(x1, x2, n)).astype(int)
        ys = np.rint(np.linspace(y1, y2, n)).astype(int)
        valid = (xs >= 0) & (xs < image_w) & (ys >= 0) & (ys < image_h)
        xs, ys = xs[valid], ys[valid]
        if xs.size == 0:
            continue
        total_on_edge += np.sum(bw_image[ys, xs])
        total_sampled += xs.size
    if total_sampled == 0:
        return 0.0
    return float(total_on_edge) / float(total_sampled)


def h2(state, bw_image, image_w, image_h):
    # h2, biased toward chains that line up with edges, not admissible.
    remaining = float(4 - len(state))
    conf = _edge_confidence(state, bw_image, image_w, image_h)
    return remaining * (1.0 - conf)


@dataclass(order=True)
class Node:
    f: float
    g: float
    state: tuple = field(compare=False)
    depth: int = field(compare=False, default=0)


# Astar for Part B and Part C quick runs, f equals g plus h, stops at max_nodes or goal score.
def astar_partBC(candidates, bw_image, image_w, image_h, use_h2=False, cost_option=1, goal_threshold=0.5, max_nodes=5000):
    start = ()
    h0 = h2(start, bw_image, image_w, image_h) if use_h2 else h1(start)
    open_list = [Node(f=h0, g=0.0, state=start, depth=0)]
    closed = set()
    best_score = 0.0
    best_rect = None
    found = False

    nodes_expanded = 0
    while open_list and nodes_expanded < max_nodes:
        node = heapq.heappop(open_list)
        if node.state in closed:
            continue
        closed.add(node.state)
        nodes_expanded += 1

        if node.depth == 4:
            rect = np.array(node.state, dtype=float)
            s = likelihoodA4(rect, bw_image, neighborhood=1)
            if s > best_score:
                best_score = s
                best_rect = rect
            if s >= goal_threshold:
                found = True
            continue

        for pt in successors(list(node.state), candidates, image_w, image_h):
            nxt = node.state + (pt,)
            if nxt in closed:
                continue
            g = g_uniform(nxt) if cost_option == 1 else g_penalty(nxt)
            h = h2(nxt, bw_image, image_w, image_h) if use_h2 else h1(nxt)
            heapq.heappush(open_list, Node(f=g + h, g=g, state=nxt, depth=len(nxt)))

    return found, best_rect, best_score


# If Astar ends weak, brute a few 4 point combos sorted by angle so we still get a number.
def run_partBC_with_fallback(
    candidates,
    edge_mask,
    image_w,
    image_h,
    use_h2=False,
    goal_threshold=0.25,
    max_nodes=4000,
    exhaustive_pool=12,
):
    found, rect, score = astar_partBC(
        candidates,
        edge_mask,
        image_w,
        image_h,
        use_h2=use_h2,
        cost_option=1,
        goal_threshold=goal_threshold,
        max_nodes=max_nodes,
    )
    if score >= goal_threshold or len(candidates) < 4:
        return found, rect, score
    pool = list(candidates)[: min(exhaustive_pool, len(candidates))]
    best_score = score
    best_rect = rect
    for quad in itertools.combinations(pool, 4):
        pts = np.array(quad, dtype=float)
        c = pts.mean(axis=0)
        ang = np.arctan2(pts[:, 1] - c[1], pts[:, 0] - c[0])
        order = np.argsort(ang)
        rect_ord = pts[order]
        s = likelihoodA4(rect_ord, edge_mask, neighborhood=1)
        if s > best_score:
            best_score = s
            best_rect = rect_ord
    found_fb = best_score >= goal_threshold
    return found_fb, best_rect, best_score


def exhaustive_quad_best_likelihood(candidates, edge_mask, exhaustive_pool=12):
    # Part D helper, best score over small combo search, see blend_exhaustive in run_partD_on_image.
    if len(candidates) < 4:
        return 0.0
    best = 0.0
    pool = list(candidates)[: min(exhaustive_pool, len(candidates))]
    for quad in itertools.combinations(pool, 4):
        pts = np.array(quad, dtype=float)
        c = pts.mean(axis=0)
        ang = np.arctan2(pts[:, 1] - c[1], pts[:, 0] - c[0])
        order = np.argsort(ang)
        rect_ord = pts[order]
        s = likelihoodA4(rect_ord, edge_mask, neighborhood=1)
        if s > best:
            best = s
    return best


# Part D BFS baseline, same successors as Astar so you can compare nodes and time.
def bfs_partD(candidates, bw_image, image_w, image_h, goal_threshold=0.5, max_nodes=20000):
    q = deque([()])
    closed = set()
    nodes_expanded = 0
    max_depth = 0
    best_score = 0.0
    best_rect = None
    found = False

    while q and nodes_expanded < max_nodes:
        state = q.popleft()
        if state in closed:
            continue
        closed.add(state)
        depth = len(state)
        max_depth = max(max_depth, depth)
        nodes_expanded += 1

        if depth == 4:
            rect = np.array(state, dtype=float)
            s = likelihoodA4(rect, bw_image, neighborhood=1)
            if s > best_score:
                best_score = s
                best_rect = rect
            if s >= goal_threshold:
                found = True
            continue

        for pt in successors(list(state), candidates, image_w, image_h):
            q.append(state + (pt,))

    return {
        "method": "BFS_h0",
        "found": found,
        "best_rect": best_rect,
        "best_score": best_score,
        "nodes_expanded": nodes_expanded,
        "search_depth": max_depth,
    }


# Part D Astar, use_h2 True means h2, False means h1, same graph as BFS.
def astar_partD(candidates, bw_image, image_w, image_h, use_h2=False, cost_option=1, goal_threshold=0.5, max_nodes=20000):
    start = ()
    h0 = h2(start, bw_image, image_w, image_h) if use_h2 else h1(start)
    open_list = [Node(f=h0, g=0.0, state=start, depth=0)]
    closed = set()
    best_score = 0.0
    best_rect = None
    found = False
    nodes_expanded = 0
    max_depth = 0

    while open_list and nodes_expanded < max_nodes:
        node = heapq.heappop(open_list)
        if node.state in closed:
            continue
        closed.add(node.state)
        nodes_expanded += 1
        max_depth = max(max_depth, node.depth)

        if node.depth == 4:
            rect = np.array(node.state, dtype=float)
            s = likelihoodA4(rect, bw_image, neighborhood=1)
            if s > best_score:
                best_score = s
                best_rect = rect
            if s >= goal_threshold:
                found = True
            continue

        for pt in successors(list(node.state), candidates, image_w, image_h):
            nxt = node.state + (pt,)
            if nxt in closed:
                continue
            g = g_uniform(nxt) if cost_option == 1 else g_penalty(nxt)
            h = h2(nxt, bw_image, image_w, image_h) if use_h2 else h1(nxt)
            heapq.heappush(open_list, Node(f=g + h, g=g, state=nxt, depth=len(nxt)))

    method_name = "Astar_h2" if use_h2 else "Astar_h1"
    return {
        "method": method_name,
        "found": found,
        "best_rect": best_rect,
        "best_score": best_score,
        "nodes_expanded": nodes_expanded,
        "search_depth": max_depth,
    }


# One image three runs, blend_exhaustive takes max of search score and that small combo score.
def run_partD_on_image(
    image_path,
    goal_threshold=0.25,
    cost_option=1,
    max_nodes=5000,
    blend_exhaustive=True,
):
    candidates, edge_mask, w, h = preprocess_candidates(image_path)
    ex_best = (
        exhaustive_quad_best_likelihood(candidates, edge_mask)
        if blend_exhaustive and len(candidates) >= 4
        else 0.0
    )
    methods = [
        ("BFS_h0", lambda: bfs_partD(candidates, edge_mask, w, h, goal_threshold, max_nodes)),
        ("Astar_h1", lambda: astar_partD(candidates, edge_mask, w, h, use_h2=False, cost_option=cost_option, goal_threshold=goal_threshold, max_nodes=max_nodes)),
        ("Astar_h2", lambda: astar_partD(candidates, edge_mask, w, h, use_h2=True, cost_option=cost_option, goal_threshold=goal_threshold, max_nodes=max_nodes)),
    ]

    rows = []
    for _, fn in methods:
        tracemalloc.start()
        t0 = time.time()
        res = fn()
        elapsed = time.time() - t0
        _, mem_peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        score_search = float(res["best_score"])
        score_out = max(score_search, ex_best) if blend_exhaustive else score_search
        found_out = res["found"] or (score_out >= goal_threshold)
        rows.append({
            "image": Path(image_path).name,
            "method": res["method"],
            "found": found_out,
            "candidates": len(candidates),
            "nodes_expanded": res["nodes_expanded"],
            "execution_time_s": round(elapsed, 4),
            "final_likelihood_score": round(score_out, 4),
            "search_depth": res["search_depth"],
            "memory_kb": round(mem_peak / 1024.0, 2),
        })
    return rows


# Loops every jpg in the folder, writes partD_results.csv, three rows per pic.
def run_partD_dataset(data_dir, output_csv="partD_results.csv", goal_threshold=0.25, cost_option=1, max_nodes=5000, blend_exhaustive=True):
    data_dir = Path(data_dir)
    images = sorted(list(data_dir.glob("*.JPG")) + list(data_dir.glob("*.jpg")))
    all_rows = []
    for img in images:
        print("Processing:", img.name)
        try:
            all_rows.extend(
                run_partD_on_image(
                    img,
                    goal_threshold=goal_threshold,
                    cost_option=cost_option,
                    max_nodes=max_nodes,
                    blend_exhaustive=blend_exhaustive,
                )
            )
        except Exception as e:
            print("  Error:", e)

    out_path = data_dir / output_csv
    if all_rows:
        fields = [
            "image", "method", "found", "candidates", "nodes_expanded",
            "execution_time_s", "final_likelihood_score", "search_depth", "memory_kb",
        ]
        with open(out_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader()
            writer.writerows(all_rows)
        print("Saved:", out_path)
    return all_rows


# Prints averages and best 5 worst 5 for Astar h1 so the report has examples.
def print_partD_summary(rows):
    if not rows:
        print("No rows to summarize.")
        return

    methods = sorted(set(r["method"] for r in rows))
    print("\nAverage metrics by method")
    for m in methods:
        sub = [r for r in rows if r["method"] == m]
        n = len(sub)
        avg_nodes = sum(r["nodes_expanded"] for r in sub) / n
        avg_time = sum(r["execution_time_s"] for r in sub) / n
        avg_score = sum(r["final_likelihood_score"] for r in sub) / n
        avg_mem = sum(r["memory_kb"] for r in sub) / n
        print(f"{m:10s} | nodes={avg_nodes:.1f} time={avg_time:.3f}s score={avg_score:.3f} mem={avg_mem:.1f}KB")

    h1_rows = [r for r in rows if r["method"] == "Astar_h1"]
    h1_rows = sorted(h1_rows, key=lambda r: (-r["final_likelihood_score"], r["execution_time_s"]))
    print("\nBest 5 cases (Astar_h1)")
    for r in h1_rows[:5]:
        print(f"{r['image']:20s} score={r['final_likelihood_score']:.3f} nodes={r['nodes_expanded']} time={r['execution_time_s']:.3f}s")

    print("\nWorst 5 cases (Astar_h1)")
    for r in h1_rows[-5:]:
        print(f"{r['image']:20s} score={r['final_likelihood_score']:.3f} nodes={r['nodes_expanded']} time={r['execution_time_s']:.3f}s")


def print_complexity_note():
    # N and b sketch only, you write the real argument in the report.
    print("\nTheoretical complexity")
    print("- Worst-case branching before pruning is about N, depth d=4.")
    print("- So worst-case search is O(N^4).")
    print("- After pruning, effective branching factor is b, so O(b^4).")
    print("- Space complexity is also O(b^4) in worst case.")

print("Notebook code loaded. No external helper file needed.")

Notebook code loaded. No external helper file needed.


In [2]:
# Quick check one image, Astar h1 vs h2 with fallback.
data_dir = Path("dbpics")
images = sorted(list(data_dir.glob("*.JPG")) + list(data_dir.glob("*.jpg")))
print("Images found:", len(images))

sample = images[0]
candidates, edge_mask, w, h = preprocess_candidates(sample)
print("Sample:", sample.name)
print("Candidate points:", len(candidates))

if len(candidates) < 4:
    print("Not enough candidates to form 4 corners yet.")
else:
    # Uses Astar then extra combo pass if score still low, see run_partBC_with_fallback in cell 1.
    found_h1, rect_h1, score_h1 = run_partBC_with_fallback(
        candidates, edge_mask, w, h, use_h2=False, goal_threshold=0.25, max_nodes=4000
    )
    found_h2, rect_h2, score_h2 = run_partBC_with_fallback(
        candidates, edge_mask, w, h, use_h2=True, goal_threshold=0.25, max_nodes=4000
    )

    print("h1 found:", found_h1, "| best score:", round(score_h1, 4))
    print("h2 found:", found_h2, "| best score:", round(score_h2, 4))

Images found: 25
Sample: 20690483_1.JPG
Candidate points: 9
h1 found: True | best score: 0.2792
h2 found: True | best score: 0.2792


In [3]:
# Same thing on five files, sanity check after edits.
for img in images[:5]:
    candidates, edge_mask, w, h = preprocess_candidates(img)
    if len(candidates) < 4:
        print(f"{img.name:20s} | cand={len(candidates):3d} | skipped (not enough points)")
        continue
    found_h1, _, score_h1 = run_partBC_with_fallback(
        candidates, edge_mask, w, h, use_h2=False, goal_threshold=0.25, max_nodes=4000
    )
    found_h2, _, score_h2 = run_partBC_with_fallback(
        candidates, edge_mask, w, h, use_h2=True, goal_threshold=0.25, max_nodes=4000
    )
    print(f"{img.name:20s} | cand={len(candidates):3d} | h1=({found_h1},{score_h1:.3f}) | h2=({found_h2},{score_h2:.3f})")

20690483_1.JPG       | cand=  9 | h1=(True,0.279) | h2=(True,0.279)
20690483_2.JPG       | cand=120 | h1=(True,0.356) | h2=(True,0.356)
20690483_3.JPG       | cand=120 | h1=(True,0.494) | h2=(True,0.494)
20690483_4.JPG       | cand=120 | h1=(True,0.459) | h2=(True,0.459)
20690483_5.JPG       | cand=120 | h1=(True,0.698) | h2=(True,0.698)


## Part D, experiments and analysis
5.1
Run this after cell 1. If blend_exhaustive is on, the CSV likelihood can be higher than search alone when search stuck at zero, see run_partD_on_image.
This section runs the required comparison:
- Breadth-First Search (or ℎ = 0)
- A* with h1
- A* with h2

For each image, it records:
- nodes expanded
- execution time
- final likelihood score
- search depth
- memory usage

In [4]:
# Part D on five pics, three algos each, takes a while.
# It prints each filename, the slow part is bfs_partD and astar_partD.
rows_quick = []
for img in images[:5]:
    print("Part D (quick): running", img.name, "...", flush=True)
    rows_quick.extend(run_partD_on_image(img, goal_threshold=0.25, cost_option=1, max_nodes=5000))

print("Rows collected:", len(rows_quick))
print_partD_summary(rows_quick)
print_complexity_note()

Part D (quick): running 20690483_1.JPG ...
Part D (quick): running 20690483_2.JPG ...
Part D (quick): running 20690483_3.JPG ...
Part D (quick): running 20690483_4.JPG ...
Part D (quick): running 20690483_5.JPG ...
Rows collected: 15

Average metrics by method
Astar_h1   | nodes=2416.4 time=9.121s score=0.000 mem=12492.3KB
Astar_h2   | nodes=2416.4 time=15.651s score=0.000 mem=11132.1KB
BFS_h0     | nodes=2416.4 time=8.038s score=0.000 mem=3422.0KB

Best 5 cases (Astar_h1)
20690483_1.JPG       score=0.000 nodes=82 time=0.015s
20690483_2.JPG       score=0.000 nodes=3000 time=11.113s
20690483_3.JPG       score=0.000 nodes=3000 time=11.154s
20690483_4.JPG       score=0.000 nodes=3000 time=11.616s
20690483_5.JPG       score=0.000 nodes=3000 time=11.709s

Worst 5 cases (Astar_h1)
20690483_1.JPG       score=0.000 nodes=82 time=0.015s
20690483_2.JPG       score=0.000 nodes=3000 time=11.113s
20690483_3.JPG       score=0.000 nodes=3000 time=11.154s
20690483_4.JPG       score=0.000 nodes=3000 ti

In [5]:
# Full folder run, saves dbpics partD_results csv, 75 rows if you have 25 pics times 3.
all_rows = run_partD_dataset(
    data_dir=Path("dbpics"),
    output_csv="partD_results.csv",
    goal_threshold=0.25,
    cost_option=1,
    max_nodes=5000
)
print("Total rows:", len(all_rows))
print("Saved file: dbpics/partD_results.csv")

Processing: 20690483_1.JPG


Processing: 20690483_2.JPG
Processing: 20690483_3.JPG
Processing: 20690483_4.JPG
Processing: 20690483_5.JPG
Processing: 20712533_1.JPG
Processing: 20712533_2.JPG
Processing: 20712533_3.JPG
Processing: 20712533_4.JPG
Processing: 20712533_5.JPG
Processing: 20713977_1.JPG
Processing: 20713977_2.JPG
Processing: 20713977_3.JPG
Processing: 20713977_4.JPG
Processing: 20713977_5.JPG
Processing: 20799262_1.JPG
Processing: 20799262_2.JPG
Processing: 20799262_3.JPG
Processing: 20799262_4.JPG
Processing: 20799262_5.JPG
Processing: 20801522_1.JPG
Processing: 20801522_2.JPG
Processing: 20801522_3.JPG
Processing: 20801522_4.JPG
Processing: 20801522_5.JPG
Saved: dbpics/partD_results.csv
Total rows: 75
Saved file: dbpics/partD_results.csv
